# Pairs trading: mean reversion in correlated cryptocurrencies

**Hypothesis:** when two highly correlated coins diverge, the log price spread reverts.
**Strategy:** z-score the spread over a 60-day window. Enter when |z| > 2 and exit when
|z| < 1, betting on reversion.

**Methodology fixes compared with the earlier version**
* **No look-ahead:** the position decided at the close of day t is earned over day t+1.
  The earlier version applied the position to the same day's return.
* **Pair selection in-sample only:** pairs are chosen on the first half of the sample. Performance
  is reported on the second half (out-of-sample) and on the full sample for reference.
* **Real costs:** each entry or exit trades both legs, so a change in position costs
  2 x 20 bps per unit. The earlier version used a hardcoded guess.
* **Multiple pairs:** all pairs are reported, not only the best one, so that the best
  pair is not reported in isolation.

Every number is printed by code.

In [ ]:
import os
import sys
from itertools import combinations

sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cryptolib import *  # noqa: F401,F403

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

UNIVERSE = ["BTCUSDT", "ETHUSDT", "ADAUSDT", "BNBUSDT", "XRPUSDT", "DOTUSDT", "MATICUSDT"]
START = "2020-01-01"
END = None
Z_LOOKBACK = 60        # days, fixed a priori
ENTRY_Z = 2.0
EXIT_Z = 1.0
CORR_THRESHOLD = 0.70  # selection threshold, applied in-sample only

## 1. Data and in-sample / out-of-sample split

In [ ]:
close_d, _ = load_universe(UNIVERSE, START, END)
px = common_window(close_d)
ret = px.pct_change().dropna()
split_date = px.index[len(px) // 2]
print(f"Common daily window: {px.index[0].date()} -> {px.index[-1].date()}")
print(f"In-sample (pair selection): up to {split_date.date()}")
print(f"Out-of-sample (performance): from {split_date.date()} onward")

## 2. Select pairs on in-sample correlations

In [ ]:
corr_in = ret.loc[:split_date].corr()
candidates = []
for a, b in combinations(px.columns, 2):
    c = corr_in.loc[a, b]
    candidates.append({"pair": f"{a}-{b}", "in-sample corr": c,
                       "selected": c > CORR_THRESHOLD})
cand = pd.DataFrame(candidates).set_index("pair").sort_values("in-sample corr", ascending=False)
print(cand.to_string())
selected = [tuple(p.split("-")) for p in cand.index[cand["selected"]]]
print(f"\nSelected pairs ({len(selected)}): {[f'{a}-{b}' for a, b in selected]}")

## 3. Pair backtest
Signal at the close of day t: z-score of the log spread (60-day trailing window, so it uses
data up to t only). The position is entered when |z| > ENTRY_Z and held until |z| < EXIT_Z.
Position +1 means long coin A and short coin B (the spread is too low). -1 is the reverse.
The held position is the previous day's signal, so there is no look-ahead.

In [ ]:
def pair_backtest(px, a, b, lookback=Z_LOOKBACK, entry=ENTRY_Z, exit_=EXIT_Z,
                  cost_rate=COST_RATE):
    spread = np.log(px[a]) - np.log(px[b])
    mu = spread.rolling(lookback).mean()
    sd = spread.rolling(lookback).std()
    z = (spread - mu) / sd

    pos = np.zeros(len(z))
    cur = 0.0
    for i, zi in enumerate(z.values):
        if np.isnan(zi):
            pos[i] = cur
            continue
        if cur == 0.0:
            if zi > entry:
                cur = -1.0       # spread too high: short A, long B
            elif zi < -entry:
                cur = 1.0        # spread too low: long A, short B
        elif abs(zi) < exit_:
            cur = 0.0            # reverted: close
        pos[i] = cur
    pos = pd.Series(pos, index=z.index)

    held = pos.shift(1).fillna(0.0)            # decided yesterday, earned today
    r_a = px[a].pct_change()
    r_b = px[b].pct_change()
    gross = held * (r_a - r_b)
    turnover = 2.0 * held.diff().abs().fillna(0.0)   # both legs trade on every change
    cost = cost_rate * turnover
    return pd.DataFrame({"gross": gross, "turnover": turnover, "cost": cost,
                         "net": gross - cost, "position": held})


pair_results = {f"{a}-{b}": pair_backtest(px, a, b) for a, b in selected}

## 4. Out-of-sample performance of the selected pairs (net of costs)

In [ ]:
oos = {name: bt.loc[split_date:, "net"] for name, bt in pair_results.items()}
if oos:
    print(metrics_table(oos, PPY_DAILY).to_string())
else:
    print("No pairs met the in-sample correlation threshold. Lower CORR_THRESHOLD or extend the sample.")

## 5. All pairs, out-of-sample (multiple-testing check)
Picking the best pair from many is biased upward. This table runs every pair, so the spread of
outcomes is visible.

In [ ]:
all_oos = {}
for a, b in combinations(px.columns, 2):
    bt = pair_backtest(px, a, b)
    all_oos[f"{a}-{b}"] = bt.loc[split_date:, "net"]
all_table = metrics_table(all_oos, PPY_DAILY)
print(all_table.loc[["Total return", "Sharpe", "Max drawdown", "p-value (mean, iid)"]]
      .T.sort_values("Sharpe", ascending=False).to_string())
n_pos = int((all_table.loc["Sharpe"] > 0).sum())
print(f"\nPairs with positive out-of-sample Sharpe: {n_pos} of {all_table.shape[1]}")

## 6. Costs and trade count (selected pairs, out-of-sample)

In [ ]:
rows = {}
for name, bt in pair_results.items():
    t = bt.loc[split_date:]
    entries = int(((t["position"] != 0) & (t["position"].shift(1).fillna(0) == 0)).sum())
    rows[name] = {"Entries (OOS)": entries,
                  "Ann. cost drag": t["cost"].mean() * PPY_DAILY,
                  "Ann. gross return": t["gross"].mean() * PPY_DAILY,
                  "Ann. net return": t["net"].mean() * PPY_DAILY}
print(pd.DataFrame(rows).T.to_string() if rows else "No selected pairs.")

## 7. Charts

In [ ]:
if pair_results:
    fig, ax = plt.subplots(figsize=(10, 5))
    for name, bt in pair_results.items():
        (1 + bt.loc[split_date:, "net"].fillna(0)).cumprod().plot(ax=ax, label=name)
    ax.axhline(1.0, color="gray", lw=0.8)
    ax.set_title("Out-of-sample growth of $1 per selected pair, net of costs")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("pairs_oos_equity.png", dpi=150)
    plt.show()

## 8. How to read the results
* A pair supports the hypothesis only if its **out-of-sample net** mean is positive and
  significant, after costs. Section 5 shows how many pairs pass, and how a best pair compares
  with the rest.
* Daily spread trades that flip often pay the full cost each time. The cost drag in
  section 6 is the number to compare with the gross return.
* A negative result is informative: it means reversion was not strong enough to pay for
  trading in this sample.

In [ ]:
print("Summary (all numbers computed above):")
if oos:
    best_name = max(oos, key=lambda k: perf_metrics(oos[k], PPY_DAILY)["Sharpe"])
    print(f"  Best selected pair out-of-sample: {best_name} "
          f"(Sharpe {perf_metrics(oos[best_name], PPY_DAILY)['Sharpe']:.2f})")
print(f"  Pairs with positive OOS Sharpe across all pairs: {n_pos} of {all_table.shape[1]}")